# Search algorithms within Optuna

Notebook for the course [Master Hyperparameter Optimization for Tabular Learning](http://www.trainindata.com/p/master-hyperparameter-optimization-for-tabular-learning)

In this notebook, I will demo how to select the search algorithm with Optuna. We will compare the use of:

- Grid Search 
- Randomized search
- Tree-structured Parzen Estimators
- Gaussian Processes

We can select the search algorithm from the [optuna.study.create_study()](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.study.create_study.html#optuna.study.create_study) class.

<div style="
    padding: 12px 16px;
    border-left: 5px solid #2196f3;
    background-color: #eaf4fd;
    border-radius: 4px;
">
<strong>Note:</strong>
Coding agents can configure many searches automatically, but understanding the differences between these setups remains useful. This notebook highlights those nuances.
</div>

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.ensemble import RandomForestClassifier

import optuna

In [2]:
# load dataset

X, y = load_breast_cancer(return_X_y=True, as_frame=True)
y = y.map({0:1, 1:0})

X.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [3]:
# the target:
# percentage of benign (0) and malignant tumors (1)

y.value_counts() / len(y)

target
0    0.627417
1    0.372583
Name: count, dtype: float64

In [4]:
# split dataset into a train and test set

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0)

X_train.shape, X_test.shape

((398, 30), (171, 30))

## Define the objective function

This is the hyperparameter response space, the function we want to minimize.

In [5]:
# Define-by-run

def objective(trial):
    
    model = RandomForestClassifier(
        n_estimators=trial.suggest_int("rf_n_estimators", 100, 1000),
        criterion=trial.suggest_categorical("rf_criterion", ['gini', 'entropy']),
        max_depth=trial.suggest_int("rf_max_depth", 1, 4),
        min_samples_split=trial.suggest_float("rf_min_samples_split", 0.01, 1),
        random_state=10,
    )

    score = cross_val_score(model, X_train, y_train, cv=3, scoring="roc_auc")
    accuracy = score.mean()
    return accuracy

## Randomized Search

Random search evaluates hyperparameter combinations at random. The key is to test a sufficiently large number of combinations to give the algorithm a reasonable chance of finding an optimal configuration.

According to the original article on random search, 60 iterations were shown to be useful, but of course, this will depend on the complexity of the model (i.e., number of hyperparameters and low effective dimension).

[RandomSampler()](https://optuna.readthedocs.io/en/stable/reference/samplers/generated/optuna.samplers.RandomSampler.html)

In [6]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.RandomSampler(),
)

# n trials here is the number of hyperparameter
# combinations to test
study.optimize(objective, n_trials=20)

[I 2026-08-26 11:45:19,422] A new study created in memory with name: no-name-4eb27431-2f5b-4a03-8aad-97cdf15b30f2
[I 2026-08-26 11:45:19,911] Trial 0 finished with value: 0.9828145233997213 and parameters: {'rf_n_estimators': 411, 'rf_criterion': 'gini', 'rf_max_depth': 4, 'rf_min_samples_split': 0.6266163640240656}. Best is trial 0 with value: 0.9828145233997213.
[I 2026-08-26 11:45:20,914] Trial 1 finished with value: 0.9862175231538398 and parameters: {'rf_n_estimators': 797, 'rf_criterion': 'entropy', 'rf_max_depth': 4, 'rf_min_samples_split': 0.34409288305516395}. Best is trial 1 with value: 0.9862175231538398.
[I 2026-08-26 11:45:21,882] Trial 2 finished with value: 0.9836210146709284 and parameters: {'rf_n_estimators': 839, 'rf_criterion': 'entropy', 'rf_max_depth': 2, 'rf_min_samples_split': 0.5982107041649735}. Best is trial 1 with value: 0.9862175231538398.
[I 2026-08-26 11:45:22,392] Trial 3 finished with value: 0.9182198180477009 and parameters: {'rf_n_estimators': 482, 'rf

In [7]:
study.best_params

{'rf_n_estimators': 929,
 'rf_criterion': 'gini',
 'rf_max_depth': 3,
 'rf_min_samples_split': 0.21239737250752921}

In [8]:
study.best_value

0.9866191295795428

In [9]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_rf_criterion,params_rf_max_depth,params_rf_min_samples_split,params_rf_n_estimators,state
0,0,0.982815,2026-08-26 11:45:19.423163,2026-08-26 11:45:19.911458,0 days 00:00:00.488295,gini,4,0.626616,411,COMPLETE
1,1,0.986218,2026-08-26 11:45:19.911832,2026-08-26 11:45:20.914365,0 days 00:00:01.002533,entropy,4,0.344093,797,COMPLETE
2,2,0.983621,2026-08-26 11:45:20.914722,2026-08-26 11:45:21.882450,0 days 00:00:00.967728,entropy,2,0.598211,839,COMPLETE
3,3,0.918220,2026-08-26 11:45:21.882833,2026-08-26 11:45:22.391977,0 days 00:00:00.509144,entropy,2,0.690206,482,COMPLETE
4,4,0.983708,2026-08-26 11:45:22.392331,2026-08-26 11:45:22.818946,0 days 00:00:00.426615,gini,4,0.460466,367,COMPLETE
5,5,0.500000,2026-08-26 11:45:22.819264,2026-08-26 11:45:23.754152,0 days 00:00:00.934888,entropy,1,0.798035,898,COMPLETE
6,6,0.983296,2026-08-26 11:45:23.754489,2026-08-26 11:45:24.420205,0 days 00:00:00.665716,entropy,1,0.169898,571,COMPLETE
7,7,0.500000,2026-08-26 11:45:24.420549,2026-08-26 11:45:24.810453,0 days 00:00:00.389904,entropy,3,0.855304,366,COMPLETE
8,8,0.887738,2026-08-26 11:45:24.810773,2026-08-26 11:45:25.498094,0 days 00:00:00.687321,entropy,4,0.699574,651,COMPLETE
9,9,0.500000,2026-08-26 11:45:25.498407,2026-08-26 11:45:25.919196,0 days 00:00:00.420789,gini,2,0.733205,390,COMPLETE


## TPE

TPESampler is the default. You'll probably remember that for TPE, we first sample a few hyperparameter combinations at random, and then we start the sequential search, based on TPE with a certain kernel, and an acquisition function.

All these parameters are determined by default, but you can change them. More details in the [TPESampler()](https://optuna.readthedocs.io/en/stable/reference/samplers/generated/optuna.samplers.TPESampler.html) documentation.

**A note on defaults**: everything in this notebook was run against Optuna 4.9.0, the latest stable release. `TPESampler` (independent, non-multivariate) is still the implicit default in this version. Optuna 5.0 switches the default to a *multivariate* TPE with a constant-liar strategy, and also swaps the default hyperparameter-importance evaluator to PED-ANOVA. To keep results consistent across versions, it is worth defining the sampler explicitly, as we do here, instead of relying on a default that will change.

In [10]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        # the number of initial points to sample 
        # at random before doing the sequential search
        n_startup_trials=10,
        # number of initial candidates to sample to evaluate
        # the expected improvement (acquisition function)
        n_ei_candidates=24, 
    ),
)

study.optimize(objective, n_trials=20)

[I 2026-08-26 11:45:32,043] A new study created in memory with name: no-name-7601fcf9-a685-48b7-8122-8eb795d734cf
[I 2026-08-26 11:45:32,642] Trial 0 finished with value: 0.5 and parameters: {'rf_n_estimators': 554, 'rf_criterion': 'gini', 'rf_max_depth': 1, 'rf_min_samples_split': 0.9301406466217816}. Best is trial 0 with value: 0.5.
[I 2026-08-26 11:45:32,904] Trial 1 finished with value: 0.983855421686747 and parameters: {'rf_n_estimators': 212, 'rf_criterion': 'gini', 'rf_max_depth': 4, 'rf_min_samples_split': 0.44787472167792586}. Best is trial 1 with value: 0.983855421686747.
[I 2026-08-26 11:45:33,694] Trial 2 finished with value: 0.9836210146709287 and parameters: {'rf_n_estimators': 666, 'rf_criterion': 'entropy', 'rf_max_depth': 1, 'rf_min_samples_split': 0.06113816197088626}. Best is trial 1 with value: 0.983855421686747.
[I 2026-08-26 11:45:34,383] Trial 3 finished with value: 0.9834570936808458 and parameters: {'rf_n_estimators': 580, 'rf_criterion': 'entropy', 'rf_max_dep

In [11]:
study.best_params

{'rf_n_estimators': 760,
 'rf_criterion': 'entropy',
 'rf_max_depth': 4,
 'rf_min_samples_split': 0.07062125174434987}

In [12]:
study.best_value

0.9881550692566182

## Gaussian Process (GPSampler)

[GPSampler()](https://optuna.readthedocs.io/en/stable/reference/samplers/generated/optuna.samplers.GPSampler.html) is Optuna's Gaussian-process-based Bayesian optimizer. It fits a GP surrogate (Matern kernel) over the trials seen so far and picks the next point with the Log Expected Improvement acquisition function.

It works when the search space is **non-conditional** (no `if` branches deciding which parameters exist) and contains a handful of continuous or discrete parameters. It is not recommended for conditional, multi-model spaces (see the CASH notebook).

Note: `GPSampler` needs `torch` and `scipy` installed.

In [13]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.GPSampler(
        n_startup_trials=10,
    ),
)

study.optimize(objective, n_trials=20)

/var/folders/n4/2v690vyd1rd0c0rjks_7_fhc0000gn/T/ipykernel_35585/2018615473.py:3: ExperimentalWarning: GPSampler is experimental (supported from v3.6.0). The interface can change in the future.
  sampler=optuna.samplers.GPSampler(
[I 2026-08-26 11:45:48,812] A new study created in memory with name: no-name-33ff1cd5-09a3-4ce1-abde-e06816c81bd6
[I 2026-08-26 11:45:49,729] Trial 0 finished with value: 0.9862175231538398 and parameters: {'rf_n_estimators': 707, 'rf_criterion': 'gini', 'rf_max_depth': 3, 'rf_min_samples_split': 0.24775678893399788}. Best is trial 0 with value: 0.9862175231538398.
[I 2026-08-26 11:45:50,083] Trial 1 finished with value: 0.9826973198918122 and parameters: {'rf_n_estimators': 299, 'rf_criterion': 'gini', 'rf_max_depth': 3, 'rf_min_samples_split': 0.5632816477458757}. Best is trial 0 with value: 0.9862175231538398.
[I 2026-08-26 11:45:51,002] Trial 2 finished with value: 0.5 and parameters: {'rf_n_estimators': 845, 'rf_criterion': 'gini', 'rf_max_depth': 1, 'rf

In [14]:
study.best_params

{'rf_n_estimators': 1000,
 'rf_criterion': 'entropy',
 'rf_max_depth': 4,
 'rf_min_samples_split': 0.01}

In [15]:
study.best_value

0.9890451602327678

## Grid Search

[GridSampler()](https://optuna.readthedocs.io/en/stable/reference/samplers/generated/optuna.samplers.GridSampler.html)

It's very unlikely that you'll do GridSearch with Optuna. But in case you wanted to, you can. You need to add a variable with the exact hyperparameter values that you want to be tested, and pass that variable to the sampler.

In [16]:
search_space = {
    "rf_n_estimators": [100, 500, 1000],
    "rf_criterion": ['gini', 'entropy'],
    "rf_max_depth": [1, 2, 3],
    "rf_min_samples_split": [0.1, 1.0]
}


In [17]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.GridSampler(search_space),
)

study.optimize(objective)

[I 2026-08-26 11:46:03,943] A new study created in memory with name: no-name-71a0d6f5-326c-44bd-a6d1-5a83a80a0491
[I 2026-08-26 11:46:04,670] Trial 0 finished with value: 0.9870272928448488 and parameters: {'rf_n_estimators': 500, 'rf_criterion': 'entropy', 'rf_max_depth': 3, 'rf_min_samples_split': 0.1}. Best is trial 0 with value: 0.9870272928448488.
[I 2026-08-26 11:46:05,889] Trial 1 finished with value: 0.983540693385788 and parameters: {'rf_n_estimators': 1000, 'rf_criterion': 'entropy', 'rf_max_depth': 1, 'rf_min_samples_split': 0.1}. Best is trial 0 with value: 0.9870272928448488.
[I 2026-08-26 11:46:06,449] Trial 2 finished with value: 0.5 and parameters: {'rf_n_estimators': 500, 'rf_criterion': 'gini', 'rf_max_depth': 3, 'rf_min_samples_split': 1.0}. Best is trial 0 with value: 0.9870272928448488.
[I 2026-08-26 11:46:06,597] Trial 3 finished with value: 0.986938775510204 and parameters: {'rf_n_estimators': 100, 'rf_criterion': 'entropy', 'rf_max_depth': 3, 'rf_min_samples_spl

In [18]:
study.best_params

{'rf_n_estimators': 100,
 'rf_criterion': 'gini',
 'rf_max_depth': 3,
 'rf_min_samples_split': 0.1}

In [19]:
study.best_value

0.9879108269813951